# 2. Evaluate the anomaly model

This notebook tests a saved model artifact against every defect category in the configured MVTec-style dataset. It computes per-image IoU, writes visualizations, and saves a JSON report.

Training is intentionally absent here: evaluation loads `output/<category>/anomaly_model.npz`.

## Workflow

1. Load the saved memory bank and thresholds.
2. Recreate the frozen ResNet18 feature extractor.
3. Predict every non-good test image.
4. Compare masks with ground truth using IoU.
5. Save visualizations and `evaluation_results.json`.

In [6]:
import json
from pathlib import Path

from anomaly_model import (
    ResNetFeatureExtractor,
    calculate_iou,
    create_neighbor_index,
    find_ground_truth,
    get_device,
    image_paths,
    load_artifact,
    load_config,
    load_ground_truth,
    predict,
    save_prediction_visualization,
)

In [7]:
# Evaluate only this category. It must match the artifact being tested.
TARGET_CATEGORY = 'metal_nut'
config = load_config(Path('config.yaml'), category=TARGET_CATEGORY)
device = get_device()
artifact_path = config['output_dir'] / 'anomaly_model.npz'
memory_bank, thresholds, metadata = load_artifact(artifact_path)
model = ResNetFeatureExtractor(metadata['backbone_name']).to(device)
model.eval()
neighbor_index = create_neighbor_index(memory_bank, metadata['num_neighbors'])
print(f'Target category: {config["category"]}')
print(f'Loaded: {artifact_path}')
print(f'Device: {device}')
print(f'Thresholds: {thresholds}')

Target category: metal_nut
Loaded: C:\Users\Prashant Suthar\Projects\mvtec_project\src\output\metal_nut\anomaly_model.npz
Device: cpu
Thresholds: Thresholds(pixel_threshold=0.827004075050354, image_threshold=1.0)


## Evaluate defect categories

Each result stores the image path, defect type, anomaly score, image decision, IoU, and predicted boxes. The saved visualizations make false positives and localization quality easy to inspect.

In [8]:
all_results = []
test_root = config['dataset_root'] / 'test'
defect_types = sorted(path.name for path in test_root.iterdir() if path.is_dir() and path.name != 'good')

for defect_type in defect_types:
    category_results = []
    for image_path in image_paths(test_root / defect_type):
        prediction = predict(image_path, model, neighbor_index, thresholds, config, device)
        ground_truth = load_ground_truth(
            find_ground_truth(config['dataset_root'], defect_type, image_path),
            (prediction.anomaly_map.shape[1], prediction.anomaly_map.shape[0]),
            config['binary_mask_value'],
        )
        result = {
            'image_path': str(image_path),
            'defect_type': defect_type,
            'anomaly_score': prediction.anomaly_score,
            'detected': prediction.is_anomalous,
            'iou': calculate_iou(prediction.binary_mask, ground_truth),
            'predicted_boxes': [box._asdict() for box in prediction.bounding_boxes],
        }
        visualization_path = config['output_dir'] / 'visualizations' / defect_type / f'{image_path.stem}.png'
        save_prediction_visualization(prediction, ground_truth, visualization_path, config)
        category_results.append(result)
        all_results.append(result)
    if category_results:
        mean_iou = sum(item['iou'] for item in category_results) / len(category_results)
        print(f'{defect_type}: mean IoU = {mean_iou:.4f}')

bent: mean IoU = 0.4988
color: mean IoU = 0.3570
flip: mean IoU = 0.2453
scratch: mean IoU = 0.3406


In [10]:
results_path = config['output_dir'] / 'evaluation_results.json'
results_path.write_text(json.dumps(all_results, indent=4), encoding='utf-8')
overall_iou = sum(item['iou'] for item in all_results) / len(all_results) if all_results else 0.0
print(f'Overall mean IoU: {overall_iou:.4f}')
print(f'Saved results: {results_path}')

Overall mean IoU: 0.3634
Saved results: C:\Users\Prashant Suthar\Projects\mvtec_project\src\output\metal_nut\evaluation_results.json


## Interpreting the report

IoU measures pixel overlap between the predicted and ground-truth masks. A high image detection rate does not necessarily mean localization is good, so inspect IoU and the saved visualizations together.